# CatBoost with Monotone Constraints

Force CatBoost to respect known medical directions of effect on numerical features.

**Motivation**: The synthetic data was generated from the Cleveland Heart Disease dataset,
which follows established medical relationships. Encoding these as hard constraints
may improve generalization by ruling out spurious directions the model might otherwise fit.

**Constraints applied to numerical features only** (CatBoost does not support monotone constraints on cat_features):

| Feature | Direction | Reasoning |
|---|---|---|
| `age` | +1 | Older → higher disease risk |
| `bp` | +1 | Higher resting BP → higher risk |
| `cholesterol` | +1 | Higher serum cholesterol → higher risk |
| `max_hr` | −1 | Higher max HR achieved → better fitness → lower risk |
| `st_depression` | +1 | ST depression on EKG → ischemia → higher risk |

**Baseline to beat**: CatBoost default OOF AUC = 0.95540

In [6]:
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

BASELINE_AUC = 0.95540

In [7]:
KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(pd.read_csv(DATA_DIR / 'train.csv'))
test  = prep(pd.read_csv(DATA_DIR / 'test.csv'))
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
FEATURES     = NUM_FEATURES + CAT_FEATURES

X = train[FEATURES]
y = train['heart_disease'].values
X_test = test[FEATURES]

print(f'Train: {train.shape}  Test: {test.shape}')

Train: (630000, 15)  Test: (270000, 14)


## Define Constraint Sets

Three configs to test:
- **Full**: all 5 numerical features constrained
- **Confident**: only the 3 with strongest medical evidence (age, max_hr, st_depression)
- **Baseline**: no constraints (reproduced here for direct comparison)

In [8]:

# CatBoost monotone_constraints: dict of feature_name → +1 or -1
# Only numerical features; cat features are excluded (unsupported)
# NOTE: monotone_constraints requires task_type='CPU' — GPU is unsupported.

CONSTRAINTS_FULL = {
    'age':           1,   # older → more risk
    'bp':            1,   # higher BP → more risk
    'cholesterol':   1,   # higher cholesterol → more risk (weakest assumption)
    'max_hr':       -1,   # higher max HR → better fitness → less risk
    'st_depression': 1,   # ST depression → ischemia → more risk
}

CONSTRAINTS_CONFIDENT = {
    'age':           1,
    'max_hr':       -1,
    'st_depression': 1,
}

GPU_PARAMS = dict(
    iterations            = 2000,
    learning_rate         = 0.05,
    depth                 = 6,
    loss_function         = 'Logloss',
    eval_metric           = 'AUC',
    task_type             = 'GPU',
    cat_features          = CAT_FEATURES,
    random_seed           = 42,
    verbose               = 0,
)

CPU_PARAMS = dict(
    iterations            = 2000,
    learning_rate         = 0.05,
    depth                 = 6,
    loss_function         = 'Logloss',
    eval_metric           = 'AUC',
    task_type             = 'CPU',
    thread_count          = -1,   # all cores
    cat_features          = CAT_FEATURES,
    random_seed           = 42,
    verbose               = 0,
)

# (name, base_params, constraints)
configs = [
    ('no_constraints',    GPU_PARAMS, {}),
    ('constraints_3feat', CPU_PARAMS, CONSTRAINTS_CONFIDENT),
    ('constraints_5feat', CPU_PARAMS, CONSTRAINTS_FULL),
]

print('Configs to run:')
for name, _, c in configs:
    task = 'GPU' if not c else 'CPU'
    print(f'  {name} [{task}]: {c}')


Configs to run:
  no_constraints [GPU]: {}
  constraints_3feat [CPU]: {'age': 1, 'max_hr': -1, 'st_depression': 1}
  constraints_5feat [CPU]: {'age': 1, 'bp': 1, 'cholesterol': 1, 'max_hr': -1, 'st_depression': 1}


## 5-Fold CV for Each Config

In [9]:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for cfg_name, base_params, constraints in configs:
    print(f'\n=== {cfg_name} ===')
    params = {**base_params}
    if constraints:
        params['monotone_constraints'] = constraints

    oof        = np.zeros(len(y))
    test_folds = np.zeros((5, len(X_test)))
    best_iters = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx],      y[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=(X_val, y_val),
            early_stopping_rounds=50,
        )
        best_iters.append(model.best_iteration_)
        oof[val_idx]      = model.predict_proba(X_val)[:, 1]
        test_folds[fold]  = model.predict_proba(X_test)[:, 1]

        fold_auc = roc_auc_score(y_val, oof[val_idx])
        print(f'  Fold {fold+1}  AUC={fold_auc:.5f}  best_iter={model.best_iteration_}')

    cv_auc     = roc_auc_score(y, oof)
    test_preds = test_folds.mean(axis=0)
    delta      = cv_auc - BASELINE_AUC

    print(f'  OOF AUC = {cv_auc:.5f}  (delta={delta:+.5f})  '
          f'mean_best_iter={np.mean(best_iters):.0f}')

    results.append(dict(
        name       = cfg_name,
        cv_auc     = cv_auc,
        delta      = delta,
        oof        = oof,
        test_preds = test_preds,
    ))



=== no_constraints ===


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 1  AUC=0.95569  best_iter=953


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 2  AUC=0.95471  best_iter=1050


...

  Fold 3  AUC=0.95550  best_iter=1107


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 4  AUC=0.95506  best_iter=1146


Default metric period is 5 because AUC is/are not implemented for GPU


  Fold 5  AUC=0.95586  best_iter=1012
  OOF AUC = 0.95536  (delta=-0.00004)  mean_best_iter=1054

=== constraints_3feat ===
  Fold 1  AUC=0.95479  best_iter=531
  Fold 2  AUC=0.95385  best_iter=608
  Fold 3  AUC=0.95465  best_iter=642
  Fold 4  AUC=0.95417  best_iter=592
  Fold 5  AUC=0.95498  best_iter=668
  OOF AUC = 0.95449  (delta=-0.00091)  mean_best_iter=608

=== constraints_5feat ===
  Fold 1  AUC=0.95455  best_iter=318
  Fold 2  AUC=0.95363  best_iter=540
  Fold 3  AUC=0.95437  best_iter=487
  Fold 4  AUC=0.95393  best_iter=524
  Fold 5  AUC=0.95470  best_iter=432
  OOF AUC = 0.95423  (delta=-0.00117)  mean_best_iter=460


## Summary

In [10]:
print(f'=== Summary ===')
print(f'{"Config":<25} {"OOF AUC":>10}  {"Delta":>8}')
print('-' * 47)
print(f'{"catboost_default (baseline)":<25} {BASELINE_AUC:>10.5f}  {"":>8}')
for r in results:
    marker = ' <-- best' if r['cv_auc'] == max(x['cv_auc'] for x in results) else ''
    print(f'{r["name"]:<25} {r["cv_auc"]:>10.5f}  {r["delta"]:>+8.5f}{marker}')

=== Summary ===
Config                       OOF AUC     Delta
-----------------------------------------------
catboost_default (baseline)    0.95540          
no_constraints               0.95536  -0.00004 <-- best
constraints_3feat            0.95449  -0.00091
constraints_5feat            0.95423  -0.00117


## Submit Best Config (if better than baseline)

In [11]:
ON_KAGGLE = Path('/kaggle/working').exists()

for r in results:
    if r['name'] == 'no_constraints':
        continue  # already submitted as catboost_default

    label = f'cat_{r["name"]}'
    fname = f'submissions/{label}.csv'
    sub   = ss.copy()
    sub['Heart Disease'] = r['test_preds']
    sub.to_csv(fname, index=False)

    desc = f'{label} | cv_auc={r["cv_auc"]:.4f}'
    print(f'\n{label}  CV={r["cv_auc"]:.5f}  delta={r["delta"]:+.5f}')

    if ON_KAGGLE:
        print(f'Submit: kaggle competitions submit -c playground-series-s6e2 -f {fname} -m "{desc}"')
    else:
        result = subprocess.run(
            ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
             '-f', fname, '-m', desc],
            capture_output=True, text=True
        )
        status = 'submitted' if 'successfully' in result.stdout.lower() else result.stdout.strip()[:120]
        print(f'  {status}')


cat_constraints_3feat  CV=0.95449  delta=-0.00091
  submitted

cat_constraints_5feat  CV=0.95423  delta=-0.00117
  submitted


## Conclusion

In [12]:
best = max(results, key=lambda r: r['cv_auc'])
print(f'Best config: {best["name"]}  AUC={best["cv_auc"]:.5f}  delta={best["delta"]:+.5f}')
print()
if best['cv_auc'] > BASELINE_AUC + 0.0001:
    print('Monotone constraints improved AUC meaningfully. '
          'Medical prior is consistent with the generative model.')
elif best['cv_auc'] > BASELINE_AUC:
    print('Marginal CV gain — within public LB noise (~0.0003). '
          'Constraints may help on private LB (189k rows).')
else:
    print('No improvement. The generative model either does not strictly '
          'follow these medical monotonicity assumptions, or CatBoost '
          'already respects them without explicit enforcement.')

Best config: no_constraints  AUC=0.95536  delta=-0.00004

No improvement. The generative model either does not strictly follow these medical monotonicity assumptions, or CatBoost already respects them without explicit enforcement.
